# Installations and Imported Packages

In [1]:
#!pip install wandb

In [2]:
import numpy as np
import pandas as pd
import os
import random
from kaggle_secrets import UserSecretsClient
import wandb
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import torch

import warnings
warnings.filterwarnings("ignore")

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

# Weights & Biases Setup and Test run

In [3]:
# try:
#     wandb.login()
#     print("Login Successful")
# except Exception as e:
#     print(f"Login Failed: {e}")

In [4]:
# for run in range(5):
#     # Start a new wandb run to track this script.
#     run = wandb.init(
#         # Set the wandb entity where your project will be logged (generally your team name).
#         entity="21f2000660-dl-gen-ai-project-26-t1",
#         # Set the wandb project where this run will be logged.
#         project="21f2000660-t12026",
#         # Track hyperparameters and run metadata.
#         config={
#             "learning_rate": 0.02,
#             "architecture": "CNN",
#             "dataset": "CIFAR-100",
#             "epochs": 10,
#         },
#     )

# # Simulate training.
# epochs = 10
# offset = random.random() / 5
# for epoch in range(2, epochs):
#     acc = 1 - 2**-epoch - random.random() / epoch - offset
#     loss = 2**-epoch + random.random() / epoch + offset

#     # Log metrics to wandb.
#     run.log({"acc": acc, "loss": loss})

# # Finish the run and upload any remaining data.
# run.finish()

# Milestone 1

This milestone focuses on understanding the dataset and establishing a baseline performance through **exploratory data analysis (EDA)** and simple **heuristic-based methods** using `librosa`.

---

## Suggested Readings
- [Hugging Face Audio Course](https://huggingface.co/learn/audio-course/en/chapter0/introduction)
- [Librosa Documentation](https://librosa.org/doc/main/core.html#audio-loading)

---

## Instructions
Use this notebook to answer **all Milestone-1 questions**.

---

## Resources
- Notebook Link:  
  https://colab.research.google.com/drive/1m6UczhxQIke_raWSqukSWuiKbIVt7MMb?usp=sharing  

- Competition Link:  
  https://www.kaggle.com/competitions/jan-2026-dl-gen-ai-project/


In [5]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [6]:
# CONFIGURATION
path = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems'
DATA_ROOT = path # Enter dataset path

with os.scandir(path) as entries:
    directories = [entry.name for entry in entries]

GENRES = sorted(directories) # Make the list of all genres available (alphabetical order)
print(GENRES)

STEMS = {'drums':'drums.wav', 'vocals':'vocals.wav', 'bass':'bass.wav', 'other':'other.wav'} # Write here stems file name
print(STEMS)

STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0 #Enter index as per Q10.

['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
{'drums': 'drums.wav', 'vocals': 'vocals.wav', 'bass': 'bass.wav', 'other': 'other.wav'}


In [7]:
def build_dataset(root_dir, val_split=0.17, seed=42):
    # Initialize empty dictionaries
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}

    print(train_dataset)

    rng = random.Random(seed)

    # ------------------- write your code here -------------------------------

        # Iterate through Genres
        # Check: if genre folder exists
        # CHECK : Completeness (Does it have all stems?)
        # CHECK : Corruption (Is any file too small? (less than 4kb))
        # size checks
        # Stratified Shuffle Split
     #-------------------------------------------------------------------------

    # Limits given in the questions
    limit_1 = 4 * 1024
    limit_2 = 5.0491 * 1024 * 1024
    limit_3 = 5.0493 * 1024 * 1024

    corrupted_songs = 0
    small_songs = 0
    large_songs = 0
    
    # Helper function to populate dict
    def add_to_dict(target_dict, genre, song_paths):
        for song_path in song_paths:
            for stem_key, stem_file in STEMS.items():
                target_dict[genre][stem_key].append(os.path.join(song_path, stem_file))

    for genre in GENRES:
        genre_path = os.path.join(root_dir, genre)
        
        # Check: if genre folder exists
        if not os.path.isdir(genre_path):
            continue
            
        # Get all song folders
        song_folders = sorted([f.path for f in os.scandir(genre_path) if f.is_dir()])
        valid_songs = []

        print(song_folders[:3])

        for song in song_folders:
            is_valid = True
            
            # All stems for a song should be available, if not invalid 
            for stem_file in STEMS.values():
                file_path = os.path.join(song, stem_file)
                
                if not os.path.exists(file_path):
                    is_valid = False
                    continue
                
                file_size = os.path.getsize(file_path)
                
                if file_size < limit_1:
                    corrupted_songs += 1
                    is_valid = False
                
                if file_size < limit_2:
                    small_songs += 1
                    
                if file_size > limit_3:
                    large_songs += 1

            if is_valid:
                valid_songs.append(song)

        # Stratified Shuffle Split
        rng.shuffle(valid_songs)
        
        split_index = int(len(valid_songs) * (1 - val_split))
        train_paths = valid_songs[:split_index]
        val_paths = valid_songs[split_index:]
        
        add_to_dict(train_dataset, genre, train_paths)
        add_to_dict(val_dataset, genre, val_paths)

    print('Q1',corrupted_songs + small_songs)
    print('Q2',abs(large_songs - small_songs))
    print('Q3',abs(len(train_dataset['reggae']['drums']) - len(val_dataset['country']['vocals'])))

    return train_dataset, val_dataset

tr, val = build_dataset(DATA_ROOT)

print(val[GENRE_TO_TEST])

{'blues': {'drums': [], 'vocals': [], 'bass': [], 'other': []}, 'classical': {'drums': [], 'vocals': [], 'bass': [], 'other': []}, 'country': {'drums': [], 'vocals': [], 'bass': [], 'other': []}, 'disco': {'drums': [], 'vocals': [], 'bass': [], 'other': []}, 'hiphop': {'drums': [], 'vocals': [], 'bass': [], 'other': []}, 'jazz': {'drums': [], 'vocals': [], 'bass': [], 'other': []}, 'metal': {'drums': [], 'vocals': [], 'bass': [], 'other': []}, 'pop': {'drums': [], 'vocals': [], 'bass': [], 'other': []}, 'reggae': {'drums': [], 'vocals': [], 'bass': [], 'other': []}, 'rock': {'drums': [], 'vocals': [], 'bass': [], 'other': []}}
['/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/blues/blues.00000', '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/blues/blues.00001', '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/blues/blues.00002']
['/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/classical/classical.00000', '/k

In [8]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    """
    Input:
        dataset_dict: The dictionary structure {genre: {stem: [paths...]}}
    Output:
        df: Pandas DataFrame containing details of all files with silence >= 5s
    """
    records = []
    # ------------------- write your code here -------------------------------

    total_files = 0    # ---- COUNT TOTAL FILES ----



        # Load Audio

        # Find Non-Silent Intervals

        # CASE A: Fully silent
        # CASE B: START silence
        # CASE C: END silence
        # CASE D: MIDDLE silence

        # Store result
        # if max_silence >= threshold_sec:
        #     records.append({
        #         "Genre": genre,
        #         "Stem": stem_name,
        #         "Duration": round(total_duration, 2),
        #         "Max_Silence_Sec": round(max_silence, 2),
        #         "Silence_Location": ", ".join(silence_type),
        #         "File_Path": file_path
        #     })
    #-------------------------------------------------------------------------

    # Iterate through the nested dictionary
    for genre, stems in dataset_dict.items():
        for stem_name, file_paths in stems.items():
            for file_path in file_paths:
                total_files += 1
                
                # Load Audio
                try:
                    y, _ = librosa.load(file_path, sr=sr)
                    total_duration = librosa.get_duration(y=y, sr=sr)
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")
                    continue

                # Find Non-Silent Intervals
                # intervals is a list of [start, end] sample indices of NON-silence
                intervals = librosa.effects.split(y, top_db=top_db)
                
                silence_durations = []
                silence_type = []

                # CASE A: Fully silent
                if len(intervals) == 0:
                    max_silence = total_duration
                    silence_type.append("Full")
                else:
                    # CASE B: START silence (from 0 to first non-silent start)
                    if intervals[0][0] > 0:
                        start_silence = intervals[0][0] / sr
                        silence_durations.append(start_silence)
                        if start_silence >= threshold_sec:
                             silence_type.append("Start")

                    # CASE C: END silence (from last non-silent end to total length)
                    if intervals[-1][1] < len(y):
                        end_silence = (len(y) - intervals[-1][1]) / sr
                        silence_durations.append(end_silence)
                        if end_silence >= threshold_sec:
                             silence_type.append("End")

                    # CASE D: MIDDLE silence (gaps between intervals)
                    for i in range(len(intervals) - 1):
                        silence_gap = (intervals[i+1][0] - intervals[i][1]) / sr
                        silence_durations.append(silence_gap)
                        if silence_gap >= threshold_sec:
                             silence_type.append("Middle")
                    
                    # Determine max silence for this file
                    if silence_durations:
                        max_silence = max(silence_durations)
                    else:
                        max_silence = 0.0

                # Store result
                if max_silence >= threshold_sec:
                    records.append({
                        "Genre": genre,
                        "Stem": stem_name,
                        "Duration": round(total_duration, 2),
                        "Max_Silence_Sec": round(max_silence, 4),
                        "Silence_Location": ", ".join(set(silence_type)), # use set to avoid duplicates
                        "File_Path": file_path
                    })
    df = pd.DataFrame(records)
    return df


# --- EXECUTION ---
# Pass your 'tr' (training) dictionary here.
# Ensure 'tr' is defined from your previous build_dataset code.
df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)

# --- RESULTS ANALYSIS ---

# ------------------- write your code here -------------------------------

print('Q4',len(df_silence))
print('Q5',len(df_silence[df_silence['Stem'] == 'vocals']))
print('Q6',df_silence[df_silence['Stem'] == 'vocals']['Max_Silence_Sec'].mean())
print('Q7',len(df_silence[(df_silence['Genre'] == 'jazz') & (df_silence['Stem'] == 'drums')]))
print('Q8',len(df_silence[(df_silence['Genre'] == 'jazz') & (df_silence['Stem'] == 'drums') & (df_silence['Silence_Location'] == 'Middle')]))
print('Q9',len(df_silence[(df_silence['Genre'] == 'jazz') & (df_silence['Stem'] == 'drums') & (df_silence['Max_Silence_Sec'] >= 10.0)]))
#-------------------------------------------------------------------------
# Hint: Create a pivot Table: Count by Genre vs Stem

pivot_table = df_silence.pivot_table(index='Genre', columns='Stem', values='Max_Silence_Sec', aggfunc='count', fill_value=0)
print("\n--- Pivot Table ---")
print(pivot_table)

Q4 680
Q5 304
Q6 12.590756578947369
Q7 24
Q8 13
Q9 7

--- Pivot Table ---
Stem       bass  drums  other  vocals
Genre                                
blues        17     22      5      43
classical    68     57      5      69
country      16     16      2      16
disco         8      2      3      18
hiphop       20      3     22       6
jazz         25     24      1      70
metal         6      2      1      40
pop          10      5      2       4
reggae        5      4      7      12
rock         10      7      1      26


In [9]:
stems_audio = []
try:
    for key in STEM_KEYS:
    # ------------------- write your code here -------------------------------
    # Load audio (Duration 5.0s for speed/consistency)
        file_path = tr[GENRE_TO_TEST][key][SONG_INDEX]

        y, _ = librosa.load(file_path, sr=SR, duration=DURATION)
        stems_audio.append(y)
    #-------------------------------------------------------------------------

    print("Audio loaded successfully.")
except NameError:
    print("ERROR: 'tr' dictionary not found. Please run build_dataset() first.")
except IndexError:
    print(f"ERROR: Song index {SONG_INDEX} out of range for genre {GENRE_TO_TEST}.")
except Exception as e:
    print(f"ERROR: {e}")

Audio loaded successfully.


In [10]:
# ------------------- write your code here -------------------------------
# Stack them into a numpy array (Shape: 4 x Samples)
stems_stack = np.vstack(stems_audio)

# Mix the stems by summing them element-wise
mix_raw = np.sum(stems_stack, axis=0)

# Calculate RMS Amplitude MANUALLY
rms_val = np.sqrt(np.mean(mix_raw**2))

#Peak Normalization
max_val = np.max(np.abs(mix_raw))

if max_val > 0:
    mix_norm = mix_raw / max_val
else:
    mix_norm = mix_raw

# VALIDATION
assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."
#------------------------------------------------------------------------

print('Q10',len(mix_raw))
print('Q11',rms_val)
print('Q12',max_val)

Q10 110250
Q11 0.102124155
Q12 0.58938766


# DummyModel Submission for Registration

In [11]:
testData = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv")
path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"

with os.scandir(path) as entries:
    directories = [entry.name for entry in entries]

ids = [f"{i:04d}" for i in range(1, len(testData)+1)]
genres = random.choices(directories, k=3020)

submission = pd.DataFrame({
    'id': ids,
    'genre' : genres
})

submission.head()

submission.to_csv('/kaggle/working/submission.csv', index=False)